# NB26 — Interaction Test and Cofactor Jackknife

**Exploratory post-hoc sensitivity** — labelled exploratory; cannot shift H1 classification.

**Part A — Formal interaction test:**
Fit a joint PGLS model including both resistance and cofactor densities as predictors:
`mean_levins_B_std ~ resistance_density_z + cofactor_density_z`
Test whether β_cofactor differs significantly from β_resistance using a z-contrast.
This directly tests the claim that the two subcategories have different associations
with niche breadth, beyond what separate PGLS models (NB03) already show.

**Part B — Cofactor jackknife:**
For each of the 7 cofactor KOs in turn, remove that KO, recompute per-Mb cofactor
density for the remaining 6, run PGLS, and record β. If any single KO drives the
result, removing it should flip the sign or eliminate significance.

**Outputs:**
- `data/interaction_test_results.csv`
- `data/cofactor_jackknife_results.csv`
- `figures/cofactor_jackknife_forest.png`

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

DATA    = Path('../data')
FIGS    = Path('../figures')
sys.path.insert(0, str(Path('../scripts')))
from pgls_utils import run_pgls
import dendropy

TREE_PATH = DATA / 'gtdb_bac_genus_pruned.tree'

## Block 0 — Load gene list and bac_base

In [2]:
gene_df  = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
bac_base = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')

resist_kos   = list(gene_df[gene_df['primary_category'] == 'Resistance/Detoxification']['KO'])
cofactor_kos = list(gene_df[gene_df['primary_category'] == 'Cofactor Biosynthesis']['KO'])

print('Resistance KOs (n=%d):' % len(resist_kos), resist_kos)
print('Cofactor KOs  (n=%d):' % len(cofactor_kos), cofactor_kos)

Resistance KOs (n=106): ['K00520', 'K00537', 'K01135', 'K01551', 'K01782', 'K01784', 'K01799', 'K01809', 'K01814', 'K03087', 'K03088', 'K03297', 'K03304', 'K03325', 'K03446', 'K03543', 'K03585', 'K03712', 'K03741', 'K03892', 'K03893', 'K05606', 'K07233', 'K07240', 'K07552', 'K07665', 'K07785', 'K07786', 'K07787', 'K07788', 'K07789', 'K07796', 'K07797', 'K07798', 'K07799', 'K07803', 'K07810', 'K08151', 'K08153', 'K08160', 'K08161', 'K08167', 'K08170', 'K08221', 'K08355', 'K08356', 'K08365', 'K08721', 'K09771', 'K11326', 'K11741', 'K11811', 'K11923', 'K12151', 'K13283', 'K14470', 'K14588', 'K15549', 'K15725', 'K15726', 'K15727', 'K16264', 'K17686', 'K18131', 'K18138', 'K18139', 'K18141', 'K18142', 'K18145', 'K18146', 'K18147', 'K18299', 'K18302', 'K18303', 'K18307', 'K18324', 'K18893', 'K18898', 'K18899', 'K18901', 'K18902', 'K18903', 'K18908', 'K18924', 'K18925', 'K18975', 'K18989', 'K18990', 'K19057', 'K19576', 'K19578', 'K19591', 'K19594', 'K19595', 'K19784', 'K21905', 'K21906', 'K219

## Block 1 — Spark: per-genus per-KO presence for resistance and cofactor sets

Need per-genus count of resistance KOs and cofactor KOs separately, plus genome size.
Cache to `data/nb26_category_ko_counts.parquet`.

**Note:** Same taxonomy join uncertainty as NB25 Block 1. Verify schema in JupyterHub.

In [3]:
CACHE = DATA / 'nb26_category_ko_counts.parquet'

if CACHE.exists():
    cat_counts = pd.read_parquet(CACHE)
    print('Loaded from cache:', cat_counts.shape)
    print(cat_counts.head())
else:
    try:
        from berdl_utils import get_spark_session
        spark = get_spark_session()
    except ImportError:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName('nb26_category').getOrCreate()

    all_target_kos = resist_kos + cofactor_kos
    quoted_kos = ','.join(f"'{k}'" for k in all_target_kos)

    # Use f-string with doubled braces for literal curly braces in regex
    sql = f"""
        WITH gene_ko AS (
            SELECT g.genome_id,
                   REGEXP_EXTRACT(TRIM(ko_item), '(K[0-9]{{5}})', 1) AS ko
            FROM kbase.ke_pangenome.gene g
            JOIN kbase.ke_pangenome.bakta_annotations ann
              ON g.gene_id = ann.gene_cluster_id
            LATERAL VIEW EXPLODE(SPLIT(ann.kegg_orthology_id, ',')) t AS ko_item
            WHERE ann.kegg_orthology_id IS NOT NULL
              AND TRIM(ann.kegg_orthology_id) != ''
        ),
        gene_ko_filtered AS (
            SELECT genome_id, ko FROM gene_ko
            WHERE ko IN ({quoted_kos})
              AND ko IS NOT NULL AND ko != ''
        ),
        genome_tax AS (
            SELECT 
                genome_id,
                LOWER(SPLIT(gtdb_taxonomy_id, ';')[5]) AS genus_lower
            FROM kbase.ke_pangenome.genome
            WHERE gtdb_taxonomy_id IS NOT NULL
        )
        SELECT
            gt.genus_lower,
            gkf.ko,
            COUNT(DISTINCT gkf.genome_id) AS n_genomes_with_ko
        FROM gene_ko_filtered gkf
        JOIN genome_tax gt ON gkf.genome_id = gt.genome_id
        WHERE gt.genus_lower IS NOT NULL
        GROUP BY gt.genus_lower, gkf.ko
    """

    cat_counts = spark.sql(sql).toPandas()
    cat_counts.attrs = {}   # clear non‑serializable metadata
    cat_counts.to_parquet(CACHE)
    print('Computed and cached:', cat_counts.shape)

Loaded from cache: (74587, 3)
         genus_lower      ko  n_genomes_with_ko
0     g__rhodococcus  K03446                 45
1  g__flavobacterium  K03543                141
2      g__lentimonas  K07787                  8
3         g__niallia  K07240                 18
4     g__gca-2726245  K07787                  4


## Block 2 — Build per-genus category density columns

In [4]:
# Defensive: strip GTDB g__ prefix if present (no-op if genome_metadata.genus is already clean)
cat_counts['genus_lower'] = (
    cat_counts['genus_lower']
    .str.replace(r'^g__', '', regex=True)
    .str.strip()
)

# Pivot: genus × KO → aggregate to genus × category
bac_genera = set(bac_base['genus_lower'])
piv = cat_counts[cat_counts['genus_lower'].isin(bac_genera)].copy()

# Sum KO counts per genus per category
resist_set   = set(resist_kos)
cofactor_set = set(cofactor_kos)

piv['category'] = piv['ko'].apply(
    lambda k: 'resistance' if k in resist_set else ('cofactor' if k in cofactor_set else 'other')
)

# Correct approach: binary presence per KO, then count per category
piv['present'] = (piv['n_genomes_with_ko'] > 0).astype(int)
agg_binary = piv.groupby(['genus_lower', 'category'])['present'].sum().unstack(fill_value=0)

print('Category counts per genus (head):')
print(agg_binary.head())

# Merge with bac_base for genome_mb
bac = bac_base.set_index('genus_lower')
common = sorted(set(agg_binary.index) & set(bac.index))
agg_binary = agg_binary.loc[common]
bac_al     = bac.loc[common]

genome_mb = bac_al['mean_genome_mb'].values

n_resist   = agg_binary.get('resistance', pd.Series(0, index=agg_binary.index)).values
n_cofactor = agg_binary.get('cofactor',   pd.Series(0, index=agg_binary.index)).values

resist_density   = n_resist   / genome_mb
cofactor_density = n_cofactor / genome_mb

df_joint = pd.DataFrame({
    'genus_lower':          common,
    'mean_levins_B_std':    bac_al['mean_levins_B_std'].values,
    'resist_density_z':    (resist_density   - resist_density.mean())   / (resist_density.std()   + 1e-12),
    'cofactor_density_z':  (cofactor_density - cofactor_density.mean()) / (cofactor_density.std() + 1e-12),
})
print(f'\nJoint DataFrame: {df_joint.shape}')
print(df_joint.describe())


Category counts per genus (head):
category             cofactor  resistance
genus_lower                              
abiotrophia                 1           4
abyssicoccus                2           8
acaryochloris               2          11
acetanaerobacterium         0           6
acetatifactor               0           9

Joint DataFrame: (1570, 4)
       mean_levins_B_std  resist_density_z  cofactor_density_z
count        1570.000000      1.570000e+03        1.570000e+03
mean            0.243926      3.032252e-16       -5.430900e-17
std             0.150907      1.000319e+00        1.000319e+00
min             0.000831     -1.910881e+00       -1.408721e+00
25%             0.123786     -7.309026e-01       -6.317110e-01
50%             0.222344     -1.808971e-01       -4.781935e-02
75%             0.346922      4.986051e-01        5.843509e-01
max             0.747430      5.811785e+00        5.879835e+00


## Block 3 (Part A) — Joint PGLS and interaction test

Model: `mean_levins_B_std ~ resist_density_z + cofactor_density_z`

Compare β_cofactor vs β_resistance using a z-test on the difference of regression coefficients.
This is the formal test of whether the two categories have statistically different associations.

In [5]:
# Joint model with both category predictors
res_joint = run_pgls(
    df_joint, TREE_PATH,
    response='mean_levins_B_std',
    predictors=['resist_density_z', 'cofactor_density_z'],
)  # taxon_col defaults to 'genus_lower'

# Extract coefficients (multi-predictor model returns dicts)
def _extract(res, focal):
    if 'beta' in res and len(res.get('predictors', [])) == 1:
        return res['beta'], res['SE'], res['p_value']
    return res['betas'][focal], res['SEs'][focal], res['p_values'][focal]

b_R, se_R, p_R = _extract(res_joint, 'resist_density_z')
b_C, se_C, p_C = _extract(res_joint, 'cofactor_density_z')

print('Joint PGLS results:')
print(f'  beta_resistance = {b_R:.5f}  SE={se_R:.5f}  p={p_R:.4e}')
print(f'  beta_cofactor   = {b_C:.5f}  SE={se_C:.5f}  p={p_C:.4e}')

# z-test for difference of coefficients (conservative: assumes independence)
delta_b = b_C - b_R
se_diff = np.sqrt(se_R**2 + se_C**2)
z_diff  = delta_b / se_diff
p_diff  = 2 * stats.norm.sf(abs(z_diff))

print(f'\nContrast β_cofactor − β_resistance = {delta_b:.5f}')
print(f'  z = {z_diff:.3f},  p (two-tailed, conservative) = {p_diff:.4e}')
print('  Note: SE of difference assumes independence of estimates.')

# Also run separate baseline models for comparison
res_R = run_pgls(df_joint, TREE_PATH, response='mean_levins_B_std',
                 predictors=['resist_density_z'])
res_C = run_pgls(df_joint, TREE_PATH, response='mean_levins_B_std',
                 predictors=['cofactor_density_z'])

print(f'\nSeparate models (for comparison):')
print(f'  resistance only: beta={res_R["beta"]:.5f}  p={res_R["p_value"]:.4e}')
print(f'  cofactor   only: beta={res_C["beta"]:.5f}  p={res_C["p_value"]:.4e}')


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


Joint PGLS results:
  beta_resistance = 0.00303  SE=0.00413  p=4.6347e-01
  beta_cofactor   = -0.02469  SE=0.00504  p=1.0756e-06

Contrast β_cofactor − β_resistance = -0.02772
  z = -4.254,  p (two-tailed, conservative) = 2.1008e-05
  Note: SE of difference assumes independence of estimates.


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(



Separate models (for comparison):
  resistance only: beta=-0.00451  p=2.4304e-01
  cofactor   only: beta=-0.02332  p=6.9862e-07


In [6]:
# Save interaction test results
int_rows = [
    {'model': 'joint_resist',   'predictor': 'resist_density_z',   'beta': b_R, 'SE': se_R, 'p': p_R},
    {'model': 'joint_cofactor', 'predictor': 'cofactor_density_z', 'beta': b_C, 'SE': se_C, 'p': p_C},
    {'model': 'contrast',       'predictor': 'cofactor - resistance',
     'beta': delta_b, 'SE': se_diff, 'p': p_diff},
    {'model': 'separate_resist',   'predictor': 'resist_density_z',   'beta': res_R['beta'], 'SE': res_R['SE'], 'p': res_R['p_value']},
    {'model': 'separate_cofactor', 'predictor': 'cofactor_density_z', 'beta': res_C['beta'], 'SE': res_C['SE'], 'p': res_C['p_value']},
]
int_df = pd.DataFrame(int_rows)
int_df.to_csv(DATA / 'interaction_test_results.csv', index=False)
print(int_df.to_string())

               model              predictor      beta        SE             p
0       joint_resist       resist_density_z  0.003026  0.004127  4.634679e-01
1     joint_cofactor     cofactor_density_z -0.024694  0.005043  1.075557e-06
2           contrast  cofactor - resistance -0.027721  0.006517  2.100756e-05
3    separate_resist       resist_density_z -0.004506  0.003858  2.430417e-01
4  separate_cofactor     cofactor_density_z -0.023320  0.004681  6.986206e-07


## Block 4 — Spark: per-genus per-KO density for each cofactor KO (jackknife cache)

The Spark query from Block 1 already gives us per-KO presence counts for cofactor KOs.
Use `cat_counts` filtered to cofactor KOs to build the jackknife presence matrix.

In [7]:
# Extract per-genus per-cofactor-KO presence from the cached cat_counts
cof_piv = cat_counts[
    (cat_counts['genus_lower'].isin(bac_genera)) &
    (cat_counts['ko'].isin(cofactor_set))
].copy()

cof_piv['present'] = (cof_piv['n_genomes_with_ko'] > 0).astype(int)

# Pivot: (genus × cofactor_KO) presence matrix
cof_matrix = cof_piv.pivot_table(index='genus_lower', columns='ko', values='present', fill_value=0)
print('Cofactor KO presence matrix:', cof_matrix.shape)
print('Cofactor KOs in matrix:', list(cof_matrix.columns))

# Align with bac_base for genome_mb and B_std
bac = bac_base.set_index('genus_lower')
common_cof = sorted(set(cof_matrix.index) & set(bac.index))
cof_matrix = cof_matrix.loc[common_cof]
bac_cof    = bac.loc[common_cof]

C_genome_mb = bac_cof['mean_genome_mb'].values
C_B_std     = bac_cof['mean_levins_B_std'].values
C_genera    = list(common_cof)

# Full-set cofactor density (all available KOs in matrix)
C_P = cof_matrix.values.astype(float)
full_cofactor_density = C_P.sum(axis=1) / C_genome_mb
full_density_z = (full_cofactor_density - full_cofactor_density.mean()) / full_cofactor_density.std()

df_full = pd.DataFrame({'genus_lower': C_genera,
                         'density_z': full_density_z,
                         'mean_levins_B_std': C_B_std})
res_full = run_pgls(df_full, TREE_PATH, response='mean_levins_B_std',
                    predictors=['density_z'])
beta_full = res_full['beta']
p_full    = res_full['p_value']
se_full   = res_full['SE']
print(f'\nFull cofactor set ({C_P.shape[1]} KOs in matrix): beta={beta_full:.5f}  SE={se_full:.5f}  p={p_full:.4e}')


Cofactor KO presence matrix: (1297, 4)
Cofactor KOs in matrix: ['K01772', 'K02225', 'K03635', 'K22225']


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(



Full cofactor set (4 KOs in matrix): beta=-0.02780  SE=0.00453  p=1.1213e-09


## Block 5 (Part B) — Jackknife: leave-one-KO-out for cofactor set

In [8]:
cof_ko_list = list(cof_matrix.columns)
jackknife_rows = []

for excl_ko in cof_ko_list:
    keep_kos = [k for k in cof_ko_list if k != excl_ko]
    if len(keep_kos) == 0:
        continue

    density_j = cof_matrix[keep_kos].values.sum(axis=1) / C_genome_mb

    if density_j.std() < 1e-10:
        print(f'Excluding {excl_ko}: degenerate (all-zero density after removal)')
        jackknife_rows.append({'excluded_ko': excl_ko, 'n_kos_remaining': len(keep_kos),
                                'beta': np.nan, 'SE': np.nan, 'p': np.nan,
                                'beta_sign_change': True, 'significant': False})
        continue

    density_z = (density_j - density_j.mean()) / density_j.std()
    df_j = pd.DataFrame({'genus_lower': C_genera, 'density_z': density_z,
                          'mean_levins_B_std': C_B_std})
    try:
        res_j = run_pgls(df_j, TREE_PATH, response='mean_levins_B_std',
                         predictors=['density_z'])
        b_j  = res_j['beta']
        se_j = res_j['SE']
        p_j  = res_j['p_value']
    except Exception as e:
        print(f'PGLS failed for exclude={excl_ko}: {e}')
        b_j, se_j, p_j = np.nan, np.nan, np.nan

    sign_change = (np.sign(b_j) != np.sign(beta_full)) if not np.isnan(b_j) else True
    significant = (p_j < 0.05) if not np.isnan(p_j) else False

    jackknife_rows.append({
        'excluded_ko':      excl_ko,
        'n_kos_remaining':  len(keep_kos),
        'beta':             b_j,
        'SE':               se_j,
        'p':                p_j,
        'beta_sign_change': sign_change,
        'significant':      significant,
    })
    print(f'  exclude {excl_ko}: beta={b_j:.5f}  SE={se_j:.5f}  p={p_j:.4e}  sign_change={sign_change}  sig={significant}')

jk_df = pd.DataFrame(jackknife_rows)
jk_df.to_csv(DATA / 'cofactor_jackknife_results.csv', index=False)

n_sign_change   = int(jk_df['beta_sign_change'].sum())
n_insignificant = int((~jk_df['significant']).sum())
print(f'\nSign changes: {n_sign_change}/{len(jk_df)}')
print(f'Lost significance: {n_insignificant}/{len(jk_df)}')
print(f'Full-set beta = {beta_full:.5f}  SE={se_full:.5f}  p = {p_full:.4e}')


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  exclude K01772: beta=-0.01633  SE=0.00441  p=2.2230e-04  sign_change=False  sig=True


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  exclude K02225: beta=-0.02704  SE=0.00444  p=1.4183e-09  sign_change=False  sig=True


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  exclude K03635: beta=-0.02872  SE=0.00443  p=1.3054e-10  sign_change=False  sig=True


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  exclude K22225: beta=-0.02775  SE=0.00478  p=8.2449e-09  sign_change=False  sig=True

Sign changes: 0/4
Lost significance: 0/4
Full-set beta = -0.02780  SE=0.00453  p = 1.1213e-09


## Block 6 — Forest plot: jackknife β values

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BLUE, RED, GREY = '#0072B2', '#CC3311', '#999999'
GREEN = '#009E73'

fig, ax = plt.subplots(figsize=(9, max(5, len(jk_df) * 0.7 + 2)))

kos    = list(jk_df['excluded_ko'])
betas  = jk_df['beta'].values
ses    = jk_df['SE'].fillna(0).values
y      = np.arange(len(kos))

colours = [RED if sc else BLUE for sc in jk_df['beta_sign_change']]

ax.errorbar(betas, y, xerr=1.96 * ses, fmt='none', color=GREY, lw=1.5, capsize=4, zorder=1)
ax.scatter(betas, y, color=colours, s=80, zorder=2)

# Full-set reference line
ax.axvline(beta_full, color=GREEN, lw=2, ls='--', label=f'Full set β = {beta_full:.4f}')
ax.axvline(0, color='black', lw=0.8, ls='-', alpha=0.4)

ax.set_yticks(y)
ax.set_yticklabels([f'Exclude {ko}' for ko in kos], fontsize=10)
ax.set_xlabel('PGLS β (cofactor density → niche breadth)', fontsize=11)
ax.set_title('Cofactor jackknife: leave-one-KO-out sensitivity\n'
             '(red = sign change; blue = same sign as full set)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(FIGS / 'cofactor_jackknife_forest.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: figures/cofactor_jackknife_forest.png')

Figure saved: figures/cofactor_jackknife_forest.png


## Block 7 — Summary report

In [10]:
print('=' * 60)
print('PART A — JOINT PGLS INTERACTION TEST')
print('=' * 60)
print(f'  beta_resistance (joint model): {b_R:.5f}  SE={se_R:.5f}  p={p_R:.4e}')
print(f'  beta_cofactor   (joint model): {b_C:.5f}  SE={se_C:.5f}  p={p_C:.4e}')
print(f'  Contrast (cofactor - resistance): Δβ = {delta_b:.5f}')
print(f'  z = {z_diff:.3f},  p (conservative, two-tailed) = {p_diff:.4e}')

print()
print('=' * 60)
print('PART B — COFACTOR JACKKNIFE')
print('=' * 60)
print(f'  Full-set β = {beta_full:.5f}  p = {p_full:.4e}')
print()
print(jk_df[['excluded_ko','beta','SE','p','beta_sign_change','significant']].to_string(index=False))
print(f'\n  Sign changes: {n_sign_change}/{len(jk_df)}')
print(f'  Lost significance: {n_insignificant}/{len(jk_df)}')

if n_sign_change == 0 and n_insignificant == 0:
    print('\n  VERDICT: Cofactor signal is robust — no single KO drives the result.')
elif n_sign_change > 0:
    sign_ko = jk_df.loc[jk_df['beta_sign_change'], 'excluded_ko'].tolist()
    print(f'\n  VERDICT: Removing {sign_ko} flips the sign — these KOs are influential.')
    print('  Interpret cofactor result with caution; discuss leverage KOs explicitly.')
else:
    insig_ko = jk_df.loc[~jk_df['significant'], 'excluded_ko'].tolist()
    print(f'\n  VERDICT: Removing {insig_ko} eliminates significance but does not flip sign.')
    print('  Signal is directionally robust but magnitude depends partly on these KOs.')

print()
print('Saved:')
print('  data/interaction_test_results.csv')
print('  data/cofactor_jackknife_results.csv')
print('  figures/cofactor_jackknife_forest.png')

PART A — JOINT PGLS INTERACTION TEST
  beta_resistance (joint model): 0.00303  SE=0.00413  p=4.6347e-01
  beta_cofactor   (joint model): -0.02469  SE=0.00504  p=1.0756e-06
  Contrast (cofactor - resistance): Δβ = -0.02772
  z = -4.254,  p (conservative, two-tailed) = 2.1008e-05

PART B — COFACTOR JACKKNIFE
  Full-set β = -0.02780  p = 1.1213e-09

excluded_ko      beta       SE            p  beta_sign_change  significant
     K01772 -0.016335 0.004412 2.222963e-04             False         True
     K02225 -0.027044 0.004435 1.418338e-09             False         True
     K03635 -0.028725 0.004433 1.305449e-10             False         True
     K22225 -0.027754 0.004784 8.244868e-09             False         True

  Sign changes: 0/4
  Lost significance: 0/4

  VERDICT: Cofactor signal is robust — no single KO drives the result.

Saved:
  data/interaction_test_results.csv
  data/cofactor_jackknife_results.csv
  figures/cofactor_jackknife_forest.png


In [13]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("../data")
CONFOUND = DATA / "confound_results"

# 1. Load PGLS input (genus metadata + niche breadth)
pgls = pd.read_csv(DATA / "01_pgls_input_bacteria.csv")

# 2. Load cobalamin KO lists from cached KEGG module JSON
def load_module_kos(module_id):
    p = CONFOUND / f"kegg_module_{module_id}.json"
    if not p.exists():
        return set()
    data = json.loads(p.read_text())
    kos = set()

    # Try to extract a list of entries from the JSON
    entries = []

    # Case 1: top-level "definition" exists
    if "definition" in data:
        defn = data["definition"]
        if isinstance(defn, dict):
            # If definition is a dict, try to get "entry" list
            entries = defn.get("entry", [])
        elif isinstance(defn, list):
            # If definition is a list, use it directly
            entries = defn
        else:
            # If definition is a string, split it
            entries = str(defn).split()
    else:
        # Case 2: no "definition" key – maybe the whole file is a list
        entries = data if isinstance(data, list) else []

    # Extract KO IDs (strings starting with "K")
    for entry in entries:
        # entry could be a string, a dict, or something else; convert to str and split
        for token in str(entry).split():
            if token.startswith("K"):
                kos.add(token)

    return kos

cob_kos = load_module_kos("M00122") | load_module_kos("M00924")

# 3. Load genus KO presence (from pathway completeness run)
ko_df = pd.read_parquet(DATA / "genus_ko_presence_all.parquet")
# Only cobalamin KOs
cob_df = ko_df[ko_df["ko"].isin(cob_kos)].copy()

# Completeness = fraction of pathway KOs present per genus
# A KO is "present" if n_genomes_with_ko >= 1
cob_present = (
        cob_df[cob_df["n_genomes_with_ko"] >= 1]
        .groupby("genus_lower")["ko"]
        .nunique()
        .rename("n_kos_present")
        .reset_index()
    )

In [14]:
cob_present

,genus_lower,n_kos_present
0,0-14-0-80-60-11,10
1,0-14-3-00-41-53,6
2,01-full-45-10b,5
3,02-full-45-11b,1
4,13-2-20cm-66-19,9
...,...,...
5124,zoogloea,9
5125,zooshikella,2
5126,zth2,10
5127,zymobacter,4
